<div dir="rtl" style="text-align:right">

# شرح کیوبیت

*کرهٔ بلاخ، دوران‌های گروه $SU(2)$، تحول زمانی و اندازه‌گیری. روایتی راهنمامحور و محاسبه‌محور با زبان ولفرام*

**ممد بهرامی**، مؤسسهٔ تحقیقاتی ولفرام، ایالات متحدهٔ آمریکا

> این یک نمونهٔ کوتاه است برای آزمودنِ نمایشِ راست‌به‌چپ، ریاضیات و سلول‌های کد در Colab.

</div>

In [ ]:
# === Setup: free Wolfram Engine + wolframclient (RUN THIS CELL FIRST) ===
# Colab edition of "Qubit Explained". Every computation is the exact Wolfram Language
# expression from the original, evaluated on the FREE Wolfram Engine. The Engine is free
# for personal/learning/pre-production use; you only need a (free) Wolfram ID:
#   https://www.wolfram.com/engine/free-license
#
# Colab VMs are temporary, so this one-time setup (install + activate) runs once per
# session: a few minutes the first time you run the notebook.
import os, sys, subprocess, getpass, glob

IN_COLAB = "google.colab" in sys.modules

def _sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

# 1) Install the free Wolfram Engine (Ubuntu .deb, ~2.3 GB; once per VM).
if not os.path.exists("/usr/local/bin/wolframscript"):
    if IN_COLAB or sys.platform.startswith("linux"):
        _sh("wget -q https://wolfr.am/wolfram-engine.deb -O /tmp/we.deb")
        _sh("sudo apt-get -qq install -y /tmp/we.deb")
    else:
        print("Install the free Wolfram Engine from https://www.wolfram.com/engine/ "
              "(on macOS: brew install --cask wolfram-engine).")
else:
    print("Wolfram Engine already installed.")

# 2) Python client libraries.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wolframclient", "pexpect"], check=True)

# 3) One-time activation with your FREE Wolfram ID (first run on a given VM).
def _licensed():
    try:
        r = subprocess.run(["wolframscript", "-code", "$LicenseType"],
                           capture_output=True, text=True, timeout=180)
        return r.returncode == 0 and bool(r.stdout.strip()) and "$Failed" not in r.stdout
    except Exception:
        return False

def _activate():
    if _licensed():
        print("Wolfram Engine already activated."); return
    import pexpect
    wid = input("Wolfram ID (email): ")
    pw  = getpass.getpass("Wolfram password: ")
    # Activation only prompts under -activate; an unactivated engine running a
    # -code command just prints "run with the -activate option" and exits.
    c = pexpect.spawn("wolframscript -activate", encoding="utf-8", timeout=600)
    i = c.expect(["Wolfram ID:", "already activated", pexpect.EOF])
    if i == 0:
        c.sendline(wid)
        c.expect("Password:"); c.sendline(pw)
        c.expect(pexpect.EOF)
    print((c.before or "").strip()[-400:])
_activate()

# 4) Start a kernel session and define the helpers used throughout the notebook.
from wolframclient.evaluation import WolframLanguageSession
from wolframclient.language import wlexpr

def _kernel_path():
    # wolframclient's auto-search misses the Engine's install location on Colab, so
    # ask the (now-licensed) engine for $InstallationDirectory and point at its kernel.
    try:
        r = subprocess.run(["wolframscript", "-code", "$InstallationDirectory"],
                           capture_output=True, text=True, timeout=120)
        inst = r.stdout.strip().strip('"')
        for sub in ("Executables", "MacOS"):       # Linux uses Executables, macOS uses MacOS
            k = os.path.join(inst, sub, "WolframKernel")
            if os.path.exists(k):
                return k
    except Exception:
        pass
    for k in (["/usr/bin/WolframKernel", "/usr/local/bin/WolframKernel"]
              + sorted(glob.glob("/usr/local/Wolfram/*/*/Executables/WolframKernel"))
              + sorted(glob.glob("/opt/Wolfram/*/*/Executables/WolframKernel"))):
        if os.path.exists(k):
            return k
    return None

_K = _kernel_path()
print("Kernel:", _K or "(auto-detect)")
session = WolframLanguageSession(_K) if _K else WolframLanguageSession()
IMG_DIR = "qis-figs"; os.makedirs(IMG_DIR, exist_ok=True)

def wl(code):
    """Evaluate Wolfram Language `code` and print the result as InputForm text."""
    wrapped = ("Module[{qisRes=(\n" + code + "\n)}, If[ByteCount[qisRes]>200000,"
               " \"<<\"<>ToString[Head[qisRes]]<>\" object>>\", ToString[qisRes, InputForm]]]")
    out = session.evaluate(wlexpr(wrapped))
    if isinstance(out, str) and out != "Null":
        print(out)
    # no return value: prevents Jupyter from echoing the result a second time

def _display_file(path):
    try:
        from IPython.display import Image, display; display(Image(filename=path))
    except Exception:
        print("saved:", path)

def wlshow(code, name):
    """Render a Wolfram graphic to qis-figs/<name>.png and show it inline."""
    p = os.path.join(IMG_DIR, name + ".png")
    session.evaluate(wlexpr('Export["' + p.replace("\\", "\\\\") + '", (\n' + code + '\n), ImageResolution->144]'))
    _display_file(p)

def wlgif(frames, name):
    """Render a list/Table of Wolfram frames to an animated GIF and show it inline."""
    p = os.path.join(IMG_DIR, name + ".gif")
    session.evaluate(wlexpr('Export["' + p.replace("\\", "\\\\") + '", (' + frames + '),'
                            ' "AnimationRepetitions"->Infinity, "DisplayDurations"->0.12]'))
    _display_file(p)


<div dir="rtl" style="text-align:right">

## بخشِ یکم: بردارهای حالت و هندسهٔ بلاخ

### عددهای مختلط همچون دو درجهٔ آزادی

یک عدد مختلط را می‌توان به‌صورت $z=x+i\, y$ نوشت، که در آن $x$ و $y$ عددهای حقیقی‌اند و $i$ واحد موهومی است که با $i^{2}=-1$ تعریف می‌شود. همین عدد را می‌توان به شکلِ قطبی نیز نوشت: $z=r\, e^{i\, \varphi }$، که در آن $r=\lvert z\rvert$ قدرمطلق $z$ است و $\varphi$ آرگومان آن. کوتاه آنکه یک عدد مختلط دو درجهٔ آزادیِ حقیقی در خود دارد.

</div>

<div dir="rtl" style="text-align:right">

مزدوجِ مختلطِ $3+2i$ را محاسبه کنید:

</div>

In [ ]:
wl(r"""
Conjugate[3 + 2 I]
""")

3 - 2*I


<div dir="rtl" style="text-align:right">

$3+2i$ را به شکلِ قطبی بیابید:

</div>

In [ ]:
wl(r"""
AbsArg[3 + 2 I]
""")

{Sqrt[13], ArcTan[2/3]}


<div dir="rtl" style="text-align:right">

بررسی کنید که شکلِ قطبی همان عدد مختلطِ اصلی است:

</div>

In [ ]:
wl(r"""
#1 E^(I #2) & @@ AbsArg[3 + 2 I] == 3 + 2 I
""")

True


<div dir="rtl" style="text-align:right">

چنان‌که گفته شد، عدد مختلطِ $3+2i$ دو درجهٔ آزادی دارد: $\{3,2\}$. با همین‌ها می‌توان شکلِ قطبیِ عدد مختلط را ساخت.

</div>

In [ ]:
wl(r"""
ToPolarCoordinates[{3, 2}]
""")

{Sqrt[13], ArcTan[2/3]}
